# ProyectoIA · ResNet50 con papa balanceada

En Colab elige GPU. En Mi unidad → ProyectoIA-datos coloca:
- plantvillage-dataset.zip
- plantvillage_particiones.csv
- campo_entrenamiento_balanceado.zip

Si Drive convirtió el CSV en hoja de cálculo, la primera celda te pedirá subir el CSV original.
El notebook guarda los resultados en la subcarpeta resnet50_papa_balanceada.

El experimento conserva las 23 clases, las particiones de PlantVillage, la semilla y la
receta anterior. Mantiene las 1.782 fotos de tomate y usa 300 fotos de campo de papa
por condición: sana, tizón temprano y tizón tardío. La prueba de Pakistán está reservada
para evaluar el modelo exportado y nunca se carga en este notebook.


In [ ]:
from pathlib import Path, PurePosixPath
from tempfile import TemporaryDirectory
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
import tensorflow as tf
from google.colab import drive, files
from sklearn.metrics import accuracy_score, classification_report, f1_score

tf.keras.utils.set_random_seed(123)
assert tf.config.list_physical_devices("GPU"), "Selecciona GPU en el entorno de Colab."
drive.mount("/content/drive")
source = Path("/content/drive/MyDrive/ProyectoIA-datos")
run_dir = source / "resnet50_papa_balanceada"
local = TemporaryDirectory(prefix="proyectoia-", dir="/content")
work = Path(local.name)
manifest_path = source / "plantvillage_particiones.csv"
if not manifest_path.is_file():
    print("Sube únicamente plantvillage_particiones.csv.")
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).lower().endswith(".csv"):
        raise ValueError("Selecciona solo el CSV de particiones.")
    from io import BytesIO
    manifest = pd.read_csv(BytesIO(next(iter(uploaded.values()))))
else:
    manifest = pd.read_csv(manifest_path)
field_path = source / "campo_entrenamiento_balanceado.zip"
if not field_path.is_file():
    raise FileNotFoundError(f"Sube campo_entrenamiento_balanceado.zip a {source}")
clases = sorted(manifest["class_name"].unique())
assert len(clases) == 23
groups = manifest.groupby("leaf_group")["split"].nunique()
assert groups.max() == 1, "Hay grupos de hojas en más de una partición."

# Limita solo entrenamiento para evitar que las clases con miles de fotos dominen.
frames = {split: manifest[manifest["split"] == split] for split in ("train", "validation", "test")}
train_parts = [
    group.sample(n=min(len(group), 600), random_state=123)
    for _, group in frames["train"].groupby("class_name")
]
frames["train"] = pd.concat(train_parts, ignore_index=True)

labels = {name: i for i, name in enumerate(clases)}
paths = {}
with zipfile.ZipFile(source / "plantvillage-dataset.zip") as archive:
    for split, frame in frames.items():
        image_paths = []
        for entry in frame["zip_member"]:
            member = PurePosixPath(entry)
            target = work / split / member.parent.name / member.name
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(entry) as zipped, target.open("wb") as dest:
                shutil.copyfileobj(zipped, dest)
            image_paths.append(str(target))
        paths[split] = image_paths
        print(split, len(frame), "imágenes")


field_paths = []
field_labels = []
with zipfile.ZipFile(field_path) as archive:
    for entry in archive.namelist():
        member = PurePosixPath(entry)
        if len(member.parts) != 2 or member.parent.name not in labels or member.suffix != ".jpg":
            continue
        target = work / "field" / member.parent.name / member.name
        target.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(entry) as zipped, target.open("wb") as dest:
            shutil.copyfileobj(zipped, dest)
        field_paths.append(str(target))
        field_labels.append(member.parent.name)
if not field_paths:
    raise ValueError("El ZIP de campo no contiene imágenes de las clases esperadas.")
paths["train"].extend(field_paths)
frames["train"] = pd.concat(
    [frames["train"], pd.DataFrame({"class_name": field_labels})], ignore_index=True
)
field_counts = pd.Series(field_labels).value_counts()
assert len(field_labels) == 2682
assert all(field_counts.get(name, 0) == 300 for name in (
    "Potato___healthy", "Potato___Early_blight", "Potato___Late_blight"
)), "La muestra de papa no está balanceada."
print("Fotos de campo añadidas al entrenamiento:", len(field_labels))
print(pd.Series(field_labels).value_counts().to_string())


In [ ]:
keras = tf.keras
layers = keras.layers
batch_size = 32

def load_image(path, label):
    rgb = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    rgb = tf.image.resize(rgb, (224, 224)) / 255.0
    rgb.set_shape((224, 224, 3))
    return rgb, label

def dataset(split):
    frame = frames[split]
    ds = tf.data.Dataset.from_tensor_slices(
        (paths[split], frame["class_name"].map(labels).to_numpy(dtype=np.int32))
    )
    if split == "train":
        ds = ds.shuffle(len(frame), seed=123)
    return ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = (dataset(split) for split in ("train", "validation", "test"))

inputs = keras.Input(shape=(224, 224, 3))
x = layers.RandomFlip("horizontal_and_vertical")(inputs)
x = layers.RandomRotation(0.1)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.RandomBrightness(0.15, value_range=(0.0, 1.0))(x)
x = layers.RandomContrast(0.15, value_range=(0.0, 1.0))(x)
x = layers.Rescaling(255)(x)
x = keras.applications.resnet50.preprocess_input(x)
base = keras.applications.ResNet50(include_top=False, weights="imagenet")
base.trainable = False
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
model = keras.Model(inputs, layers.Dense(len(clases), activation="softmax")(x))

def fit(epochs, rate):
    model.compile(optimizer=keras.optimizers.Adam(rate),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    ])

head_history = fit(15, 1e-3)
head_loss = model.evaluate(val_ds, verbose=0)[0]
head_weights = model.get_weights()

base.trainable = True
for index, layer in enumerate(base.layers):
    layer.trainable = index >= len(base.layers) - 30 and not isinstance(layer, layers.BatchNormalization)

tune_history = fit(10, 1e-5)
tune_loss = model.evaluate(val_ds, verbose=0)[0]
stage = "fine_tuning"
if tune_loss >= head_loss:
    model.set_weights(head_weights)
    stage = "cabeza_clasificadora"
print("Fase seleccionada:", stage)


In [ ]:
actual = frames["test"]["class_name"].map(labels).to_numpy()
predicted = np.argmax(model.predict(test_ds), axis=1)
metrics = {
    "model": "ResNet50",
    "num_classes": len(clases),
    "input_size": 224,
    "test_images": len(actual),
    "accuracy": float(accuracy_score(actual, predicted)),
    "f1_macro": float(f1_score(actual, predicted, average="macro")),
    "selected_stage": stage,
    "train_images": len(frames["train"]),
    "validation_images": len(frames["validation"]),
    "field_train_images": len(field_labels),
}
print(json.dumps(metrics, indent=2, ensure_ascii=False))

run_dir.mkdir(parents=True, exist_ok=True)
model.save(work / "resnet50.keras")
shutil.copyfile(work / "resnet50.keras", run_dir / "resnet50.keras")
(run_dir / "class_names.json").write_text(json.dumps(clases, indent=2, ensure_ascii=False), encoding="utf-8")
(run_dir / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
report = classification_report(actual, predicted, labels=range(len(clases)), target_names=clases, zero_division=0)
(run_dir / "classification_report.txt").write_text(report, encoding="utf-8")
print("Archivos guardados en:", run_dir)
